# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring  
**Data source:** gated FlyRank warehouse, queried directly from Hugging Face with DuckDB.

This notebook uses March 2026 as the feature window and April 2026 as the future outcome window. It does not use the June `_sample` partition.

## 1. Contract in plain words

1. **What one row means:** the source table has one daily observation for one content item under one pseudonymized client on one report date. The feature frame later aggregates to one client-content item for March.
2. **Table used:** `fact_content_daily_performance` only; no client names, domains, page URLs, or content text are needed.
3. **Time window:** March 1–31, 2026 supplies the features. April 1–30, 2026 is held forward as the outcome window.
4. **What is predicted:** whether a content item's April Google Search Console impressions are at least 20% below its March impressions. This is a decline proxy for review prioritization, not a claim that a refresh will work.
5. **Deliberately excluded:** client and content identifiers are retained only for grouping/splitting and are never model features; unavailable GSC measurements are not treated as real zeros.

**Decision moment:** after March closes and before April outcomes are observed. The label requires positive March impressions and at least 20 GSC-available daily observations in each month.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
while not (repo_root / '.env').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
env_path = repo_root / '.env'
if not env_path.is_file():
    raise FileNotFoundError('Create the ignored repository-root .env with HF_TOKEN before running this notebook.')

hf_token = None
for line in env_path.read_text(encoding='utf-8').splitlines():
    if '=' not in line or line.lstrip().startswith('#'):
        continue
    key, value = line.split('=', 1)
    if key.strip() == 'HF_TOKEN':
        hf_token = value.strip().strip(chr(34)).strip(chr(39))
        break
if not hf_token or not hf_token.startswith('hf_'):
    raise ValueError('HF_TOKEN is missing or malformed in the ignored .env; token value was not displayed.')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute('CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)', [hf_token])
del hf_token

dataset_root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
march_source = 'read_parquet(' + repr(dataset_root + '/month=2026-03/*.parquet') + ')'
april_source = 'read_parquet(' + repr(dataset_root + '/month=2026-04/*.parquet') + ')'
print('Connected to the warehouse using the ignored .env; the token is not displayed.')

Connected to the warehouse using the ignored .env; the token is not displayed.


## 2. Three March verification queries

Each check prints aggregate results only. The grain test returns at most five duplicate keys internally and does not display identifiers. Availability is explicitly tested with `IS TRUE`.

In [2]:
# Query 1 — the source grain is report date × client × content.
grain_query = '\n'.join([
    'SELECT report_date, client_hash_id, content_hash_id',
    'FROM ' + march_source,
    'GROUP BY report_date, client_hash_id, content_hash_id',
    'HAVING COUNT(*) > 1',
    'LIMIT 5',
])
duplicate_keys = con.execute(grain_query).fetchall()
print('Duplicate daily grain keys returned (query limit 5):', len(duplicate_keys))
assert len(duplicate_keys) == 0, 'The expected daily grain has duplicate keys.'

Duplicate daily grain keys returned (query limit 5): 0


In [3]:
# Query 2 — count the March slice and verify its actual date span.
counts_query = '\n'.join([
    'SELECT COUNT(*) AS row_count, MIN(report_date) AS first_date, MAX(report_date) AS last_date',
    'FROM ' + march_source,
])
march_counts = con.execute(counts_query).df()
display(march_counts)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [4]:
# Query 3 — show the number and share of March rows with observed GSC data.
availability_query = '\n'.join([
    'SELECT COUNT(*) AS all_march_rows,',
    'COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,',
    'ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 2) AS available_percent',
    'FROM ' + march_source,
])
march_availability = con.execute(availability_query).df()
display(march_availability)

,all_march_rows,gsc_available_rows,available_percent
0,9841378,3611061,36.69


## 3. Five March features and when each is knowable

The feature frame has exactly five inputs; IDs are kept separately as split context.

| Feature | Knowable when? |
| --- | --- |
| `impressions_mar` | At the end of March, after March GSC reporting is available. |
| `clicks_mar` | At the end of March, from March GSC reporting. |
| `avg_position_mar` | At the end of March, computed from March's summed GSC position and impressions. |
| `gsc_days_mar` | At the end of March, counting only March rows with `gsc_data_available IS TRUE`. |
| `days_with_impressions_mar` | At the end of March, counting available March days with positive impressions. |

The label is calculated from April impressions and is never part of the feature list. Items need at least 20 GSC-available daily observations in both windows and positive March impressions.

## 4. Named limitation

**Coverage-selection bias:** requiring 20 observed days in both months excludes sparse or newly tracked items, so the eligible feature frame does not represent every page. A decline is only a review-priority proxy; these observational data cannot show that refreshing a page causes recovery.

In [5]:
feature_sql = '\n'.join([
    'WITH march AS (',
    '  SELECT client_hash_id, content_hash_id,',
    '    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_mar,',
    '    SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_mar,',
    '    SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS clicks_mar,',
    '    SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS position_sum_mar,',
    '    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND gsc_impressions > 0) AS days_with_impressions_mar',
    '  FROM ' + march_source,
    '  GROUP BY client_hash_id, content_hash_id',
    '), april AS (',
    '  SELECT client_hash_id, content_hash_id,',
    '    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_apr,',
    '    SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_apr',
    '  FROM ' + april_source,
    '  GROUP BY client_hash_id, content_hash_id',
    ')',
    'SELECT march.client_hash_id, march.content_hash_id,',
    '  march.impressions_mar, march.clicks_mar,',
    '  march.position_sum_mar / NULLIF(march.impressions_mar, 0) AS avg_position_mar,',
    '  march.gsc_days_mar, march.days_with_impressions_mar,',
    '  CAST(april.impressions_apr <= 0.8 * march.impressions_mar AS INTEGER) AS decline_20pct_apr',
    'FROM march INNER JOIN april USING (client_hash_id, content_hash_id)',
    'WHERE march.gsc_days_mar >= 20 AND april.gsc_days_apr >= 20',
    '  AND march.impressions_mar > 0',
])
feature_frame = con.execute(feature_sql).df()
FEATURE_COLUMNS = [
    'impressions_mar', 'clicks_mar', 'avg_position_mar',
    'gsc_days_mar', 'days_with_impressions_mar',
]
TARGET = 'decline_20pct_apr'
assert len(FEATURE_COLUMNS) == 5
assert feature_frame[FEATURE_COLUMNS].notna().all().all()
print('Eligible content items:', len(feature_frame))
print('April decline-label counts (0 = no 20% decline, 1 = decline):', feature_frame[TARGET].value_counts().sort_index().to_dict())
print('Five-feature frame preview; identifiers are omitted:')
display(feature_frame[FEATURE_COLUMNS].head())

Eligible content items: 95633
April decline-label counts (0 = no 20% decline, 1 = decline): {0: 48310, 1: 47323}
Five-feature frame preview; identifiers are omitted:


,impressions_mar,clicks_mar,avg_position_mar,gsc_days_mar,days_with_impressions_mar
0,2187.0,4.0,33.057156,31,31
1,560.0,1.0,10.175000,31,31
2,111.0,1.0,16.288288,28,28
3,1000.0,2.0,15.500000,31,31
4,821.0,6.0,16.866017,31,31


## 5. Deliberate leakage trap and self-check

For the trap, add exactly one temporary column that copies the April-derived label, then score it on a client-held-out split. A perfect result is proof of leakage, not model quality. Remove the column before fitting the honest model; the reported honest score uses only the five March features.

**Self-check:** three March verification queries have visible outputs; the availability filter uses `IS TRUE`; the frame contains five pre-decision features; the label uses April; client/content IDs are context only; and the limitation is stated above.

In [6]:
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

y = feature_frame[TARGET].astype(int)
groups = feature_frame['client_hash_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feature_frame[FEATURE_COLUMNS], y, groups))
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert train_clients.isdisjoint(test_clients), 'Client groups overlap across the holdout split.'

# Intentional leak: this column is a direct copy of the future-derived label.
feature_frame['leak_probe'] = y
leaky_model = DecisionTreeClassifier(max_depth=1, random_state=42)
leaky_model.fit(feature_frame.iloc[train_idx][['leak_probe']], y.iloc[train_idx])
leaky_predictions = leaky_model.predict(feature_frame.iloc[test_idx][['leak_probe']])
print('Intentional label-copy leakage accuracy:', round(accuracy_score(y.iloc[test_idx], leaky_predictions), 3))

# Remove the leakage column; only the five March features remain eligible inputs.
feature_frame = feature_frame.drop(columns=['leak_probe'])
assert 'leak_probe' not in feature_frame.columns
honest_model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=42)
honest_model.fit(feature_frame.iloc[train_idx][FEATURE_COLUMNS], y.iloc[train_idx])
honest_predictions = honest_model.predict(feature_frame.iloc[test_idx][FEATURE_COLUMNS])
honest_probabilities = honest_model.predict_proba(feature_frame.iloc[test_idx][FEATURE_COLUMNS])[:, 1]
honest_accuracy = accuracy_score(y.iloc[test_idx], honest_predictions)
honest_ap = average_precision_score(y.iloc[test_idx], honest_probabilities)
print('Honest client-held-out accuracy (five March features):', round(honest_accuracy, 3))
print('Honest client-held-out average precision:', round(honest_ap, 3))
print('Model inputs:', FEATURE_COLUMNS)
con.close()

Intentional label-copy leakage accuracy: 1.0
Honest client-held-out accuracy (five March features): 0.672
Honest client-held-out average precision: 0.736
Model inputs: ['impressions_mar', 'clicks_mar', 'avg_position_mar', 'gsc_days_mar', 'days_with_impressions_mar']
